# Lab 1.4 · Transformer blocks

**15–20 minutes · CPU**

Read the [lab guide](README.md), then run the setup. Replace each `return None` with your implementation and rerun its check.

In [ ]:
from math import sqrt

def normalize(row):
    rms = sqrt(sum(x * x for x in row) / len(row) + 1e-8)
    return [x / rms for x in row]

def mix_prefix(rows):
    return [[sum(row[j] for row in rows[:i + 1]) / (i + 1)
             for j in range(len(rows[i]))] for i in range(len(rows))]

def feed_forward(rows):
    return [[max(0, x - y), max(0, y - x)] for x, y in rows]

inputs = [[1.0, 0.0], [0.0, 2.0], [-1.0, 1.0]]

## 1. Wire the first residual update

Implement elementwise addition of two equal-shaped lists of rows, then use it to add the supplied mixing branch to the input.

In [ ]:
def add_rows(left, right):
    return None

In [ ]:
mixed = mix_prefix([normalize(row) for row in inputs])
after_attention = add_rows(inputs, mixed)
if after_attention is not None:
    assert add_rows([[1, 2], [3, 4]], [[-1, 1], [1, -4]]) == [[0, 3], [4, 0]]
    assert all(len(row) == 2 for row in after_attention)
    print("PASS — first residual update:", after_attention)

## 2. Assemble the block and remove a branch

Complete the pipeline. Start with the input. If enabled, normalize it, mix prefixes, and add the residual. If the feed-forward branch is enabled, normalize the resulting state, apply that branch, and add its residual.

In [ ]:
def block(rows, use_attention=True, use_feed_forward=True):
    return None

In [ ]:
full = block(inputs)
if full is not None:
    only_attention = block(inputs, use_feed_forward=False)
    only_ffn = block(inputs, use_attention=False)
    assert block(inputs, False, False) == inputs
    assert only_attention == after_attention
    assert full == add_rows(only_attention, feed_forward([normalize(row) for row in only_attention]))
    changed = [[2.0, 1.0], *inputs[1:]]
    assert block(changed, use_attention=False)[1:] == only_ffn[1:]
    assert block(changed, use_feed_forward=False)[1:] != only_attention[1:]
    print("Input          Attention only        FFN only             Full")
    for row in zip(inputs, only_attention, only_ffn, full):
        print("  ".join(str([round(x, 3) for x in state]) for state in row))
    print("PASS — only the mixing branch carries earlier input changes forward")

## Wrap-up

Change the last input row. Which earlier rows can change? Where would you look if a feature changes even with the mixing branch disabled?